## 기본 LLM 체인 

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini")

llm.invoke("지구의 자전 주기는 ??")


AIMessage(content="지구의 자전 주기는 약 24시간, 즉 1일입니다. 더 정확히 말하자면, 지구가 자전하는 데 걸리는 시간은 23시간 56분 4초로, 이를 '항성일'이라고 부릅니다. 그러나 일반적으로 우리가하는 하루는 태양이 한 지평선에서 다른 지평선까지 이동하는 데 걸리는 시간인 '태양일'을 기준으로 하며, 이 시간은 약 24시간입니다.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 15, 'total_tokens': 121, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'id': 'chatcmpl-DCwATNBwRnITFtX6eQqdc8I6wlyqR', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c91fe-8b15-7790-a9cf-3bfa3f228aa6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 106, 'total_tokens': 121, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'ou

In [2]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("You are an expert in astronomy. Answer the question. <Question>: {input}")

prompt

ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='You are an expert in astronomy. Answer the question. <Question>: {input}'), additional_kwargs={})])

prompt 객체에 {"input": "지구의 자전 주기는?"} 라는 입력 값이 주어졌을 때, <Question>: {input} 부분의 {input} 위치로 "지구의 자전 주기는?" 값이 전달되어 질문에 대한 프롬프트를 완성합니다

In [3]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model = "gpt-4o-mini")
chain = prompt | llm 

chain.invoke({"input": "지구의 자전 주기는?"})

AIMessage(content='지구의 자전 주기는 약 24시간입니다. 더 정확하게는, 24시간을 기준으로 하여 23시간 56분 4초 정도입니다. 이러한 주기를 "항성일"이라고 하며, 이는 지구가 한 번 자전하며 한 별을 기준으로 한 시간입니다. 그러나 우리가 일반적으로 사용하는 24시간의 주기는 "태양일"이라고 하며, 이는 태양이 같은 위치에 돌아오는 데 걸리는 시간을 기준으로 합니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 29, 'total_tokens': 134, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'id': 'chatcmpl-DCwDfo19V6MUo6e3agMmNEfn5gHwd', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c9201-9326-78d3-a09e-1cdbf93afa46-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 29, 'output_tokens': 105, 'total_tokens': 134, 'input_token_details': {'audio': 0, 

다음과 같이 프롬프트, LLM, 문자열 출력 파서(StrOutputParser)를 연결하여 체인을 만들 수 있습니다. 이 코드의 실행 결과는 GPT-3.5 모델을 통해 생성된 "지구의 자전 주기는?"에 대한 답변입니다. StrOutputParser는 모델의 출력을 문자열 형태로 파싱하여 최종 결과를 반환합니다.

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# prompt + model + output parser
prompt = ChatPromptTemplate.from_template("You are an expert in astronomy. Answer the question. <Question>: {input}")
llm = ChatOpenAI(model="gpt-4o-mini")
output_parser = StrOutputParser()

chain = prompt | llm | output_parser

chain.invoke({"input" : "지구의 자전 주기는 ?"})

'지구의 자전 주기는 약 24시간입니다. 이는 지구가 한 번 자전하는 데 걸리는 시간으로, 일반적으로 "1일"이라고 부릅니다. 보다 정확하게 말하자면, 지구의 자전 주기는 약 23시간 56분 4초로, 이를 "항성일"이라고 합니다. 그러나 보통 우리가 사용하는 24시간은 "태양일"로, 태양이 같은 위치에 다시 도달하는 데 걸리는 시간을 기준으로 합니다.'

## 멀티 체인

여러 개의 체인을 연결하거나 병렬로 실행하여 복잡한 워크플로우를 구성하는 패턴입니다. LCEL(LangChain Expression Language)을 사용하면 체인을 파이프(|)와 RunnableParallel로 유연하게 조합할 수 있습니다
- 순차적 체인 연결: 파이프 연산자로 체인 연결
- 병렬 체인 실행: RunnableParallel로 동시 실행
- 조건부 분기: 입력에 따른 체인 선택
- 체인 간 데이터 전달: 출력을 다음 체인의 입력으로

### 기본 순차 체인

In [7]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model("gpt-4o-mini")

# 1단계: 한국어를 영어로 번역
translate_prompt = ChatPromptTemplate.from_template(
    "다음 한국어를 영어로 번역하세요: {korean_word}"
)

# 2단계: 영어 단어 설명 
explain_prompt = ChatPromptTemplate.from_template(
    "다음 영어 단어를 한국어로 자세히 설명하세요: {english_word}"
)

# 체인 1: 번역
chain1 = translate_prompt | llm | StrOutputParser()

# 체인 2: 번역
chain2 = (
    {"english_word" : chain1}
    | explain_prompt 
    | llm
    | StrOutputParser()
)

result = chain2.invoke({"korean_word": "인공지능"})

print(result)

"인공지능"은 영어로 "artificial intelligence"라고 번역됩니다. 인공지능은 컴퓨터 과학의 한 분야로, 기계나 소프트웨어가 인간의 지능을 모방하거나 이와 유사한 행동을 할 수 있도록 하는 기술을 의미합니다. 

인공지능의 주요 목표는 기계가 학습, 추론, 문제 해결, 인식, 언어 이해 등의 작업을 수행할 수 있도록 하는 것입니다. 인공지능은 크게 두 가지로 나눌 수 있습니다: 

1. **약한 인공지능 (Narrow AI)**: 특정 작업이나 문제를 해결하는 데 특화된 인공지능으로, 예를 들어 음성 인식, 이미지 인식, 추천 시스템 등 다양한 분야에서 사용됩니다.

2. **강한 인공지능 (General AI)**: 인간과 유사한 수준의 지능을 가지며, 여러 작업을 수행할 수 있는 인공지능을 의미합니다. 현재는 이론적인 개념으로 존재하며, 실제로 이러한 인공지능을 개발하는 것은 여전히 연구 단계에 있습니다.

인공지능은 우리 생활에 많은 영향을 미치고 있으며, 의료, 금융, 교통, 교육 등 다양한 산업에서 활용되고 있습니다.


### 다단계 순차 체인

In [8]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = init_chat_model("gpt-4o-mini")

# 3단계 처리: 주제 분석 → 개요 작성 → 본문 작성
analyze_prompt = ChatPromptTemplate.from_template(
    "다음 주제의 핵심 키워드 3개를 추출하세요: {topic}"
)

outline_prompt = ChatPromptTemplate.from_template(
    """
    다음 키워드를 바탕으로 글의 개요를 작성하세요:
    키워드: {keywords}
    원본 주제: {topic}
    """
)

content_prompt = ChatPromptTemplate.from_template(
    """
    다음 개요를 바탕으로 300자 내외의 글을 작성하세요:
    개요: {outline}
    """
)

chain = (
    {"topic":RunnablePassthrough()}
    | RunnablePassthrough.assign(
        keywords=analyze_prompt | llm | StrOutputParser()
    )
    | RunnablePassthrough.assign(
        outline=outline_prompt | llm | StrOutputParser()
    )
    | content_prompt
    | llm
    | StrOutputParser()
)

result = chain.invoke("기후 변화와 지속 가능한 발전")

print(result)

기후 변화는 지구의 평균 온도가 상승하는 현상으로, 주로 인간 활동에 의해 촉발된 온실가스 배출이 주요 원인입니다. 이러한 변화는 자연 생태계와 인류 사회에 심각한 영향을 미치며, 자원 고갈 및 자연 재해를 초래하고 있습니다. 따라서 지속 가능한 발전의 중요성이 더욱 부각됩니다. 지속 가능성은 사회적, 경제적, 환경적 측면에서 조화로운 발전을 추구하는 개념으로, 기후 변화 대응에서 필수적입니다.

환경 보호는 기후 변화 완화의 핵심입니다. 이를 위해 재생 가능 에너지를 확대하고, 자원 관리의 효율성을 높이는 것이 필요합니다. 또한, 대중의 인식과 참여를 증진시키기 위한 교육과 정책적 지원이 중요합니다. 궁극적으로 기후 변화와 지속 가능한 발전은 상호 의존적이며, 개인과 사회 모두가 미래 세대를 위해 지속 가능한 환경 구축에 힘써야 합니다. 이러한 노력이 모여 기후 변화에 대응하고, 건강한 지구를 다음 세대에 물려줄 수 있을 것입니다.


### 병렬 체인 실행

In [10]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

llm = init_chat_model("gpt-4o-mini")

# 세 가지 관점에서 동시 분석
positive_prompt = ChatPromptTemplate.from_template(
    "{topic}의 긍정적인 측면 3가지를 설명하세요."
)

negative_prompt = ChatPromptTemplate.from_template(
    "{topic}의 부정적인 측면 3가지를 설명하세요."
)

neutral_prompt = ChatPromptTemplate.from_template(
    "{topic}에 대한 객관적인 현황을 설명하세요."
)

parallel_chain = RunnableParallel(
    positive=positive_prompt | llm | StrOutputParser(),
    negative=negative_prompt | llm | StrOutputParser(),
    neutral=neutral_prompt | llm | StrOutputParser()
)

results = parallel_chain.invoke({"topic": "원격 근무"})

print("=== 긍정적 측면 ===")
print(results["positive"])
print("\n=== 부정적 측면 ===")
print(results["negative"])
print("\n=== 객관적 현황 ===")
print(results["neutral"])

=== 긍정적 측면 ===
원격 근무는 여러 가지 긍정적인 측면이 있습니다. 그 중에서 특히 중요한 세 가지를 설명하겠습니다.

1. **유연성 및 균형**: 원격 근무는 직원들에게 더 많은 유연성을 제공합니다. 근무 장소와 시간을 선택할 수 있어 개인의 라이프스타일과 업무를 조화롭게 조절할 수 있습니다. 이는 가족 돌봄, 개인적 관심사, 휴식 시간을 충족시키는 데 도움을 주어 일과 삶의 균형을 향상시킵니다.

2. **비용 절감**: 직원과 기업 모두에게 비용 절감 효과가 있습니다. 직원은 통근 비용, 외식비, 직장 복장비용 등을 절감할 수 있고, 기업은 사무실 공간, 관리비, 유틸리티 등 많은 운영 비용을 줄일 수 있습니다. 이러한 비용 절감은 직원의 만족도를 높이는 요소가 될 수 있습니다.

3. **생산성 향상**: 많은 연구에서 원격 근무가 직원의 생산성을 높일 수 있다는 결과가 나타났습니다. 직원들은 자신이 가장 효과적으로 일할 수 있는 환경을 조성할 수 있으며, 불필요한 회의나 사무실 내 방해 요소를 줄일 수 있습니다. 또한, 개인의 집중력을 높이고, 자신에게 맞는 근무 방식을 선택할 수 있는 점이 큰 장점입니다.

이러한 긍정적인 측면들은 원격 근무가 점점 보편화되는 이유 중 일부입니다.

=== 부정적 측면 ===
원격 근무는 많은 장점을 가지고 있지만, 다음과 같은 부정적인 측면도 존재합니다:

1. **사회적 고립감**: 원격 근무를 하다 보면 동료들과의 대면 소통이 줄어들어 사회적 고립감을 느낄 수 있습니다. 이는 팀워크와 협업에 부정적인 영향을 미칠 수 있으며, 직원의 정신적 건강에도 악영향을 미칠 수 있습니다.

2. **업무와 개인 생활의 경계 모호화**: 원격 근무는 집에서 업무를 수행하기 때문에 업무와 개인 생활의 경계가 흐릿해질 수 있습니다. 이는 업무 시간 외에도 일을 하게 되거나, 반대로 업무에 집중하지 못하는 원인이 될 수 있으며, 시간이 지남에 따라 스트레스와 불만을 초래할 수 있습니다.

3. **자기 관리의 어려움**

### 병렬 결과 통합

In [11]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

llm = init_chat_model("gpt-4o-mini")

analysis_chain = RunnableParallel(
    pros=ChatPromptTemplate.from_template("{topic}의 장점") | llm | StrOutputParser(),
    cons=ChatPromptTemplate.from_template("{topic}의 단점") | llm | StrOutputParser()
)

# 결과 통합
synthesis_prompt = ChatPromptTemplate.from_template(
    """다음 분석을 종합하여 결론을 작성하세요:

장점:
{pros}

단점:
{cons}

균형 잡힌 결론을 3문장으로 작성하세요."""
)

full_chain = (
    analysis_chain
    | synthesis_prompt
    | llm
    | StrOutputParser()
)

result = full_chain.invoke({"topic": "인공지능"})
print(result)

인공지능(AI)은 효율성, 자동화, 개인화 등 다양한 장점을 제공하며, 여러 산업에서 혁신을 이끌고 있다. 그러나 데이터 정확성 문제, 윤리적 이슈, 일자리 대체와 같은 단점도 존재하여 이러한 문제를 해결하기 위한 지속적인 연구와 논의가 필요하다. 따라서 AI의 발전은 기술적 성취뿐만 아니라 사회적 책임을 동반해야 할 것이다.


### 조건부 분기

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda

llm = init_chat_model("gpt-4o-mini")

# 언어별 다른 프롬프트
korean_prompt = ChatPromptTemplate.from_template(
    "다음 한국어 질문에 한국어로 답변하세요: {question}"
)

english_prompt = ChatPromptTemplate.from_template(
    "Answer the following question in English: {question}"
)

default_prompt = ChatPromptTemplate.from_template(
    "Please answer: {question}"
)

def detect_language(input_dict):
    question = input_dict.get("question", "")

    if any('\uac00' <= char <= '\ud7a3' for char in question):
        return "korean"
    return "english"

branch_chain = RunnableBranch(
    (lambda x: detect_language(x) == "korean", korean_prompt | llm | StrOutputParser()),
    (lambda x: detect_language(x) == "english", english_prompt | llm | StrOutputParser()),
    default_prompt | llm | StrOutputParser()
)

result_kr = branch_chain.invoke({"question": "파이썬이란 무엇인가요?"})
result_en = branch_chain.invoke({"question": "What is Python?"})

print("한국어 질문 결과:", result_kr)
print("영어 질문 결과:", result_en)



### 함수 기반 라우팅

In [14]:
from langchain_core.prompts.structured import StructuredPrompt
from langchain_core.runnables import RunnableLambda

def route_by_topic(input_dict):
    topic = input_dict.get("topic", "").lower()

    if "code" in topic or "programming" in topic:
        return "technical"
    elif "business" in topic or "strategy" in topic:
        return "business"
    else:
        return "general"
    
chain = {
    "technical" : ChatPromptTemplate.from_template(
        "기술 전문가로서 답변: {question}"
    ) | llm | StrOutputParser(),
    "business" : ChatPromptTemplate.from_template(
        "비즈니스 컨설턴트로서 답변: {question}"
    ) | llm | StrOutputParser(),
    "general" : ChatPromptTemplate.from_template(
        "일반적인 관점에서 답변: {question}"
    ) | llm | StrOutputParser()
}

def routing_chain(input_dict):
    route = route_by_topic(input_dict=input_dict)
    return chain[route].invoke(input_dict)

router = RunnableLambda(routing_chain)

result = router.invoke({
    "topic": "Python programming",
    "question": "효율적인 코드 작성 방법은?"
})

print(result)

효율적인 코드를 작성하는 것은 소프트웨어 개발의 중요한 부분입니다. 다음은 효율적인 코드 작성을 위한 몇 가지 방법입니다:

1. **명확한 요구사항 파악**: 코드 작성 전에 문제를 명확히 이해하고 요구사항을 정리합니다. 이를 통해 불필요한 기능을 줄일 수 있습니다.

2. **알고리즘 선택**: 문제에 적합한 알고리즘을 선택합니다. 효율적인 시간 복잡도와 공간 복잡도를 가진 알고리즘을 우선적으로 고려해야 합니다.

3. **코드 재사용**: 중복 코드를 피하고 재사용 가능한 모듈이나 함수로 나누어 작성합니다. 이는 유지보수를 쉽게 만듭니다.

4. **단순함 유지**: 코드는 가능한 한 단순하고 직관적으로 유지합니다. 복잡한 로직은 버그를 증가시키고 유지보수를 어렵게 합니다.

5. **주석 및 문서화**: 코드를 이해하기 쉽게 주석을 달고, 필요한 경우 명확한 문서화를 합니다. 이는팀원이나 미래의 자신을 위해 중요합니다.

6. **효율적인 데이터 구조 사용**: 상황에 맞는 데이터 구조를 선택하여 성능을 최적화합니다. 예를 들어, 검색이 빈번한 경우 해시맵을 사용할 수 있습니다.

7. **최적화의 필요성 판단**: 초기 개발 단계에서는 코드의 성능을 초기에 최적화하는 것보다 기능을 완성하는 것이 중요합니다. 나중에 병목 현상이 발견될 경우 최적화를 고려합니다.

8. **테스트 작성**: 작성한 코드에 대해 충분한 테스트를 수행하여 버그를 조기에 발견하고 수정합니다. 유닛 테스트와 통합 테스트를 활용할 수 있습니다.

9. **코드 리뷰**: 동료와 코드 리뷰를 통해 다양한 피드백을 받고, 코드를 개선할 기회를 만듭니다.

10. **최신 기술 및 트렌드 학습**: 새로운 기술, 패턴, 프레임워크를 지속적으로 학습하여 더 나은 방법으로 코드를 작성할 수 있는 능력을 개발합니다.

이러한 방법들을 통해 효율적이고 유지보수하기 쉬운 코드를 작성할 수 있습니다.


### RAG 스타일 멀티 체인

In [17]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

llm = init_chat_model("gpt-4o-mini")

# 가상의 검색 함수
def retrieve_context(query: str) -> str:
    # 실제로는 벡터스토어 검색
    return f"검색된 컨텍스트: {query}에 대한 정보..."

rag_chain = (
    RunnableParallel(
        question=RunnablePassthrough(),
        context=RunnableLambda(lambda x: retrieve_context(x["question"]))
    )
    | ChatPromptTemplate.from_template(
        """컨텍스트를 참고하여 질문에 답변하세요.

        컨텍스트: {context}

        질문: {question}

        답변:"""
    )
    | llm
    | StrOutputParser()
)

result = rag_chain.invoke({"question" : "LangChain 이란?"})
print(result)

LangChain은 언어 모델을 활용하여 다양한 애플리케이션을 구축하는 데 도움을 주는 프레임워크입니다. 이 프레임워크는 자연어 처리(NLP) 작업을 간소화하고, 언어 모델을 기반으로 한 챗봇, 정보 검색 시스템, 추천 시스템 등 다양한 기능을 구현할 수 있도록 설계되었습니다. LangChain은 다양한 데이터 소스와 통합할 수 있는 기능을 제공하며, 개발자들이 손쉽게 언어 모델을 활용할 수 있도록 도와줍니다.
